# Fuego Data Pipeline

This notebook prepares news article data from the GDELT Global Knowledge Graph for downstream analysis and LLM (Nvidia's Nemotron) processing. It uses Google BigQuery (with GoogleSQL) to retrieve articles with related to Pennsylvania, extracts the main article text using Trafilatura, filters out articles that cannot be successfully extracted or contain fewer than 300 words, and organizes the resulting dataset by publication date. The processed data is then exported as JSON files for use by the project team.

In [76]:
from google.cloud import bigquery
import html 
import pandas as pd
import numpy as np

#For filtering article links
import trafilatura

In [3]:
client = bigquery.Client()

In [ ]:
#small function to convert GoogleSQL Query to a dataframe
def query_to_df(query):
    df = client.query(query).to_dataframe()

    return df

In [ ]:
#Looking at all the columns, data type, and mode
table_ref = "gdelt-bq.gdeltv2.gkg_partitioned"

table = client.get_table(table_ref)

for field in table.schema:
    print(field.name, field.field_type, field.mode)

#Looking at GDELT's documentation anything that is v2 is just a recent version of the variable, so we just exclusively use v2

GKGRECORDID STRING NULLABLE
DATE INTEGER NULLABLE
SourceCollectionIdentifier INTEGER NULLABLE
SourceCommonName STRING NULLABLE
DocumentIdentifier STRING NULLABLE
Counts STRING NULLABLE
V2Counts STRING NULLABLE
Themes STRING NULLABLE
V2Themes STRING NULLABLE
Locations STRING NULLABLE
V2Locations STRING NULLABLE
Persons STRING NULLABLE
V2Persons STRING NULLABLE
Organizations STRING NULLABLE
V2Organizations STRING NULLABLE
V2Tone STRING NULLABLE
Dates STRING NULLABLE
GCAM STRING NULLABLE
SharingImage STRING NULLABLE
RelatedImages STRING NULLABLE
SocialImageEmbeds STRING NULLABLE
SocialVideoEmbeds STRING NULLABLE
Quotations STRING NULLABLE
AllNames STRING NULLABLE
Amounts STRING NULLABLE
TranslationInfo STRING NULLABLE
Extras STRING NULLABLE


In [55]:
# Query to preview what data looks like
# Some data are comma delimited, XML format
# Amounts contain financial data, but doesn't make much sense without consistent formatting
query_sample = """
SELECT *
FROM
  `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE
  _PARTITIONDATE BETWEEN DATE('2026-08-19') AND DATE('2026-09-19')
  AND LOWER(V2Locations) LIKE '%pittsburgh%'
LIMIT 5
"""

df_sample = query_to_df(query_sample)

display(df_sample)

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,GKGRECORDID,DATE,SourceCollectionIdentifier,SourceCommonName,DocumentIdentifier,Counts,V2Counts,Themes,V2Themes,Locations,...,GCAM,SharingImage,RelatedImages,SocialImageEmbeds,SocialVideoEmbeds,Quotations,AllNames,Amounts,TranslationInfo,Extras
0,20260819234500-721,20260819234500,1,oxygen.com,https://www.oxygen.com/crime-news/penn-state-c...,None,None,TAX_FNCACT;TAX_FNCACT_LEADER;EDUCATION;SOC_POI...,"CRIME_ILLEGAL_DRUGS,722;CRIME_ILLEGAL_DRUGS,10...","3#Pittsburgh, Pennsylvania, United States#US#U...",...,"wc:552,c12.1:30,c12.10:39,c12.12:12,c12.13:15,...",https://www.oxygen.com/sites/oxygen/files/2026...,https://www.oxygen.com/sites/oxygen/files/styl...,None,https://youtube.com/c/OxygenTV;,None,"Penn State University,99;Attorney General,319;...","2,Penn State University fraternities,65;14,peo...",None,<PAGE_LINKS>https://abc7news.com/story/penn-st...
1,20260918134500-675,20260918134500,1,engineering.com,https://www.engineering.com/novolinc-launches-...,None,None,CRISISLEX_O01_WEATHER;WB_678_DIGITAL_GOVERNMEN...,"EPU_ECONOMY_HISTORIC,456;EPU_ECONOMY_HISTORIC,...","3#Sharpsburg, Pennsylvania, United States#US#U...",...,"wc:339,c1.2:3,c1.3:1,c12.1:12,c12.10:33,c12.11...",https://www.engineering.com/wp-content/uploads...,None,None,https://youtube.com/@engineeringdotcom;,None,"Open Compute Project,2513;Startup Program,2537...","100,W/cm #xB2,30;7,mm #xB2,274;100,W/cm #xB2,7...",None,<PAGE_LINKS>https://novolinc.com/</PAGE_LINKS>...
2,20260916221500-542,20260916221500,1,wknofm.org,https://www.wknofm.org/2026-09-16/the-latest-o...,"KILL#4#person#2#Pennsylvania, United States#US...","KILL#4#person#2#Pennsylvania, United States#US...",TAX_DISEASE;TAX_DISEASE_INFECTIOUS;GENERAL_HEA...,"TAX_DISEASE_OUTBREAK,171;WB_635_PUBLIC_HEALTH,...","2#Pennsylvania, United States#US#USPA#40.5773#...",...,"wc:60,c12.1:2,c12.10:8,c12.12:4,c12.13:2,c12.1...",None,None,None,https://youtube.com/channel/UCN-zcB-YKbOBXqbd_...,None,"Scott Tong,23;Amesh Adalja,139","4,person has died,198;",None,<PAGE_LINKS>https://centerforhealthsecurity.or...
3,20260916221500-1101,20260916221500,1,wbur.org,https://www.wbur.org/hereandnow/2026/09/16/mea...,"KILL#4#person#2#Pennsylvania, United States#US...","KILL#4#person#2#Pennsylvania, United States#US...",TAX_DISEASE;TAX_DISEASE_INFECTIOUS;GENERAL_HEA...,"UNGP_FORESTS_RIVERS_OCEANS,91;TAX_FNCACT_AUTHO...","2#Pennsylvania, United States#US#USPA#40.5773#...",...,"wc:50,c12.1:2,c12.10:8,c12.12:4,c12.13:2,c12.1...",https://media.wbur.org/wp/2016/06/HereandNow-3...,None,None,https://youtube.com/user/wbur;https://youtube....,None,"Scott Tong,23;Amesh Adalja,139","4,person has died,198;",None,<PAGE_LINKS>https://centerforhealthsecurity.or...
4,20260916223000-435,20260916223000,1,wfdd.org,https://www.wfdd.org/health-safety/2026-09-16/...,"KILL#4#person#2#Pennsylvania, United States#US...","KILL#4#person#2#Pennsylvania, United States#US...",TAX_DISEASE;TAX_DISEASE_INFECTIOUS;GENERAL_HEA...,"KILL,255;CRISISLEX_T03_DEAD,255;MEDICAL,117;WB...","2#Pennsylvania, United States#US#USPA#40.5773#...",...,"wc:60,c12.1:2,c12.10:8,c12.12:4,c12.13:2,c12.1...",None,None,None,https://youtube.com/wfddradio;,None,"Scott Tong,23;Amesh Adalja,139","4,person has died,198;",None,<PAGE_LINKS>https://centerforhealthsecurity.or...


In [34]:
#Selecting themes for articles and counting the number of occurrences
#in Pittsburgh for the past month
query1 ="""
SELECT
  theme,
  COUNT(*) AS article_count
FROM (
  SELECT
    DISTINCT
      DocumentIdentifier,
      theme
  FROM (
    SELECT
      DocumentIdentifier,
      SPLIT(V2Themes, ';') AS themes
    FROM `gdelt-bq.gdeltv2.gkg_partitioned`
    WHERE
      _PARTITIONDATE BETWEEN
        DATE('2026-08-19') AND DATE('2026-09-19')
      AND (
        LOWER(AllNames) LIKE '%pittsburgh%'
        OR LOWER(V2Locations) LIKE '%pittsburgh%'
      )
  ), UNNEST(themes) AS theme
)
GROUP BY theme
ORDER BY article_count DESC
LIMIT 50;
"""

In [ ]:
df1= query_to_df(query1)

df1

#The themes are hard to understand
#Found https://data.gdeltproject.org/api/v2/guides/LOOKUP-GKGTHEMES.TXT
#However there is no more documentation on the numerical meaning
#Seems like it's inspired or some themes include something similar to World Bank Topical Taxonomy
#We will extract themes using LLM (Nvidia's Nemotron)

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,theme,article_count
0,,1379
1,"CRISISLEX_C03_WELLBEING_HEALTH,615",67
2,"MEDICAL,552",66
3,"GENERAL_HEALTH,552",66
4,"CRISISLEX_T11_UPDATESSYMPATHY,607",65
5,"TAX_DISEASE_TRAUMA,607",65
6,"SOC_SUICIDE,1138",65
7,"TAX_FNCACT_NURSE,638",65
8,"MED_EMERGENCYROOM,586",65
9,"AFFECT,410",65


In [ ]:
# Query selecting one month of Pittsburgh Data with Tones and Themes
# This will be initial input for LLM teammate to look at
# In addition, articles links to be checked because some are broken links
query2 = """
SELECT
  GKGRECORDID AS Record_ID,
  DocumentIdentifier AS Article_Link,
  theme AS Theme,
  V2Tone AS Tone
FROM
  `gdelt-bq.gdeltv2.gkg_partitioned`,
  UNNEST(SPLIT(V2Themes, ';')) AS theme
WHERE
  _PARTITIONDATE BETWEEN DATE('2026-08-19') AND DATE('2026-09-19')
  AND LOWER(V2Locations) LIKE '%pittsburgh%'
  AND V2Themes IS NOT NULL
  AND V2Themes != ''
ORDER BY
  _PARTITIONDATE DESC;
"""

df2 = query_to_df(query7)

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,Record_ID,Article_Link,Theme,Tone
0,20260918134500-675,https://www.engineering.com/novolinc-launches-...,"EPU_ECONOMY_HISTORIC,456","4.4973544973545,5.29100529100529,0.79365079365..."
1,20260918134500-675,https://www.engineering.com/novolinc-launches-...,"EPU_ECONOMY_HISTORIC,1568","4.4973544973545,5.29100529100529,0.79365079365..."
2,20260918134500-675,https://www.engineering.com/novolinc-launches-...,"WB_1921_PRIVATE_SECTOR_DEVELOPMENT,1754","4.4973544973545,5.29100529100529,0.79365079365..."
3,20260918134500-675,https://www.engineering.com/novolinc-launches-...,"WB_1921_PRIVATE_SECTOR_DEVELOPMENT,2139","4.4973544973545,5.29100529100529,0.79365079365..."
4,20260918134500-675,https://www.engineering.com/novolinc-launches-...,"WB_346_COMPETITIVE_INDUSTRIES,1754","4.4973544973545,5.29100529100529,0.79365079365..."


In [32]:
df7

,Record_ID,Article_Link,Theme,Tone
0,20260918134500-675,https://www.engineering.com/novolinc-launches-...,"EPU_ECONOMY_HISTORIC,456","4.4973544973545,5.29100529100529,0.79365079365..."
1,20260918134500-675,https://www.engineering.com/novolinc-launches-...,"EPU_ECONOMY_HISTORIC,1568","4.4973544973545,5.29100529100529,0.79365079365..."
2,20260918134500-675,https://www.engineering.com/novolinc-launches-...,"WB_1921_PRIVATE_SECTOR_DEVELOPMENT,1754","4.4973544973545,5.29100529100529,0.79365079365..."
3,20260918134500-675,https://www.engineering.com/novolinc-launches-...,"WB_1921_PRIVATE_SECTOR_DEVELOPMENT,2139","4.4973544973545,5.29100529100529,0.79365079365..."
4,20260918134500-675,https://www.engineering.com/novolinc-launches-...,"WB_346_COMPETITIVE_INDUSTRIES,1754","4.4973544973545,5.29100529100529,0.79365079365..."
...,...,...,...,...
4335,20260819131500-243,https://politicspa.com/august-19-playbook-exec...,"USPEC_POLITICS_GENERAL1,2022","-1.69491525423729,2.6634382566586,4.3583535108..."
4336,20260819131500-243,https://politicspa.com/august-19-playbook-exec...,"LEADER,1340","-1.69491525423729,2.6634382566586,4.3583535108..."
4337,20260819131500-243,https://politicspa.com/august-19-playbook-exec...,"LEADER,1815","-1.69491525423729,2.6634382566586,4.3583535108..."
4338,20260819131500-243,https://politicspa.com/august-19-playbook-exec...,"LEADER,1962","-1.69491525423729,2.6634382566586,4.3583535108..."


In [33]:
df7.to_csv("pittsburgh_gdelt_one_month_themes_tones_long.csv", index=False)

Creating the dataframes

Essentially we want to filter first based on whether an article link is working or not. Our second filter will be on word count >=300. Then to prevent too many one-to-many relationships. We'll end up with 5 dataframes: Main, Themes, People, Organizations, Locations with Record_ID as primary key

\begin{aligned}
\text{GDELT} 
&\rightarrow \text{Initial SQL Query} \\
&\rightarrow \text{Main DataFrame} \\
&\rightarrow \text{Trafilatura} \\
&\rightarrow \text{Article Text} \\
&\rightarrow \text{Word Count} \geq 300 \\
&\rightarrow \text{Filtered Articles} \\
&\rightarrow \text{Record IDs} \\
&\rightarrow
\begin{cases}
\text{Themes} \\
\text{People} \\
\text{Organizations} \\
\text{Locations}
\end{cases}
\end{aligned}

In [90]:
# Testing Trafilatura

url = "http://www.newjerseytelegraph.com/news/279308749/datameds-ai-acquires-helomics-ai-cancer-diagnostics-lab-and-precision-oncology-cro-businesses-from-axe-compute"


downloaded = trafilatura.fetch_url(url)


if downloaded:
    article_text = trafilatura.extract(downloaded)
    print(article_text)
else:
    print("Failed to download the webpage.")

ACCESS Newswire
              
16 Sep 2026, 01:45 GMT+10
          
Acquisition Provides Operational Capital to Drive Health Lives Here Campaign and Expand Services into Cancer Management
PITTSBURGH, PA / ACCESS Newswire / September 15, 2026 / DataMEDS AI, Inc. (NASDAQ:MEDS) ("DataMEDS"), a Health IT company vertically integrating health data acquisition and transfer, today announced that it completed the acquisition of artificial intelligence cancer diagnostics laboratory business Helomics Corporation ("Helomics") from Axe Compute Inc. (NASDAQ:AGPU) ("Axe Compute" or the "Company"), a neocloud AI infrastructure platform.
Under the agreement, DataMEDS acquired Axe Compute's wholly-owned subsidiary Helomics in exchange for common shares and an acquisition note of DataMEDS for a total purchase value of $1.5 million. DataMEDS received the Helomics CLIA/CAP-certified clinical laboratory, inclusive of ownership of all equipment, as well as the Predictive Oncology contract research organizat

In [152]:
#Filtering out articles that are not working and filtering articles that are too short
#300 words for now as a minimum word count
#200 words is the threshold (0.4 page long) + buffer (also need to consider descriptions in the beginning of articles)

def filter_working_articles(df, min_words=300):

    working_rows = []

    for _, row in df.iterrows():

        try:
            downloaded = trafilatura.fetch_url(row['Article_Link'])

            if not downloaded:
                continue

            article_text = trafilatura.extract(downloaded)

            if not article_text:
                continue

            word_count = len(article_text.split())

            if word_count >= min_words:
                row = row.copy()
                row['Article_Text'] = article_text
                working_rows.append(row)

        except Exception:
            continue

    return pd.DataFrame(working_rows).reset_index(drop=True)

In [100]:
# Once main dataframe is filtered we use this filter for the rest of the dataframes (Themes, Organizations, People, Locations)
# Matching Record_ID from main dataframe and if it no longer exists, we drop the row

def filter_by_main_df(df, df_main):
    valid_ids = set(df_main['Record_ID']) #grabbing df_main's record_id
    return df[df['Record_ID'].isin(valid_ids)].reset_index(drop=True) #if no match then drop

In [ ]:
# Functions that need to be created for SQL then wrapped in python function and produce dataframe needed

# Initial dataframe with everything that needs to be filtered

#THEN use filter
# Main data frame for everything else 

# Parsing for Location to create GIS, city, state, country

# Extracting themes

# Extracting People

# Extracting Organizations

In [101]:
# Generate Main Dataframe containing Date_Published, Source_Name, Article_Link, Tone 
def generate_main_df(
        start_date='2026-08-19',
        end_date='2026-09-19',
        location='pittsburgh'
):

    query = f"""
        SELECT
            GKGRECORDID AS Record_ID,

            PARSE_TIMESTAMP(
                '%Y%m%d%H%M%S',
                CAST(DATE AS STRING)
            ) AS Date_Published,

            SourceCommonName AS Source_Name,

            DocumentIdentifier AS Article_Link,

            -- Getting Article title that is stored in Extras in XML format
            REGEXP_EXTRACT(
                Extras,
                r'<PAGE_TITLE>(.*?)</PAGE_TITLE>'
            ) AS Article_Title,

            V2Tone AS Tone

        FROM
            `gdelt-bq.gdeltv2.gkg_partitioned`

        WHERE
            _PARTITIONDATE BETWEEN DATE('{start_date}')
                                AND DATE('{end_date}')
            AND LOWER(V2Locations) LIKE '%{location.lower()}%'

        ORDER BY
            DATE DESC
    """

    return query_to_df(query)

In [102]:
generate_main_df()

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,Record_ID,Date_Published,Source_Name,Article_Link,Article_Title,Tone
0,20260918134500-675,2026-09-18 13:45:00+00:00,engineering.com,https://www.engineering.com/novolinc-launches-...,NovoLINC launches MaxLINC thermal interface ma...,"4.4973544973545,5.29100529100529,0.79365079365..."
1,20260918050000-655,2026-09-18 05:00:00+00:00,birminghamstar.com,http://www.birminghamstar.com/news/279314326/c...,Community Corner: Something special,"3.99208775400108,5.25984535155548,1.2677575975..."
2,20260918000000-629,2026-09-18 00:00:00+00:00,northcentralpa.com,https://www.northcentralpa.com/features/pa-ele...,Pa. election 2026: Your complete guide to the ...,"1.05540897097625,3.69393139841689,2.6385224274..."
3,20260917211500-33,2026-09-17 21:15:00+00:00,prnewswire.com,http://www.prnewswire.com/news-releases/novoli...,"NovoLINC Launches MaxLINC, Achieving Industry-...","3.29861111111111,4.34027777777778,1.0416666666..."
4,20260917120000-543,2026-09-17 12:00:00+00:00,butlereagle.com,https://www.butlereagle.com/20260916/nonprofit...,Nonprofit claims ICE agent attempted to enter ...,"-1.31061598951507,2.22804718217562,3.538663171..."
...,...,...,...,...,...,...
60,20260819234500-721,2026-08-19 23:45:00+00:00,oxygen.com,https://www.oxygen.com/crime-news/penn-state-c...,Penn State Coke Bust: Fraternity Allegedly Sel...,"-6.21931260229133,1.47299509001637,7.692307692..."
61,20260819214500-884,2026-08-19 21:45:00+00:00,myk104.com,https://myk104.com/1902/penn-state-university-...,White People's Business: Inside The Penn State...,"-4.45246690734055,1.20336943441637,5.655836341..."
62,20260819193000-807,2026-08-19 19:30:00+00:00,newsone.com,https://newsone.com/6869657/penn-state-univers...,Inside The Penn State University Cocaine Opera...,"-4.56226880394575,1.10974106041924,5.672009864..."
63,20260819150000-400,2026-08-19 15:00:00+00:00,thespiritsbusiness.com,https://www.thespiritsbusiness.com/2026/08/bob...,Boba Pops brings alcohol-filled pearls to RTDs...,"2.60586319218241,2.93159609120521,0.3257328990..."


In [56]:
# Generate the themes dataframe
def generate_themes_df(
        start_date='2026-08-19',
        end_date='2026-09-19',
        location='pittsburgh'
):

    query = f"""
        SELECT
            GKGRECORDID AS Record_ID,

            REGEXP_REPLACE(
                theme,
                r',\\d+$',
                ''
            ) AS Theme

        FROM
            `gdelt-bq.gdeltv2.gkg_partitioned`,
            UNNEST(SPLIT(V2Themes, ';')) AS theme

        WHERE
            _PARTITIONDATE BETWEEN DATE('{start_date}')
                                AND DATE('{end_date}')
            AND LOWER(V2Locations) LIKE '%{location.lower()}%'
            AND V2Themes IS NOT NULL
            AND V2Themes != ''

        ORDER BY
            Record_ID
    """

    return query_to_df(query)

In [57]:
generate_themes_df()

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,Record_ID,Theme
0,20260819131500-243,TAX_FNCACT_EXECUTIVE
1,20260819131500-243,TAX_FNCACT_EXECUTIVE
2,20260819131500-243,TAX_FNCACT_EXECUTIVE
3,20260819131500-243,TAX_FNCACT_EXECUTIVE
4,20260819131500-243,TAX_FNCACT_PIRATES
...,...,...
4335,20260918134500-675,WB_133_INFORMATION_AND_COMMUNICATION_TECHNOLOGIES
4336,20260918134500-675,ECON_ENTREPRENEURSHIP
4337,20260918134500-675,WB_566_ENVIRONMENT_AND_NATURAL_RESOURCES
4338,20260918134500-675,WB_590_ECOSYSTEMS


In [ ]:
# Function for generating the people dataframe
def generate_people_df(
        start_date='2026-08-19',
        end_date='2026-09-19',
        location='pittsburgh'
):

    query = f"""
        SELECT
            GKGRECORDID AS Record_ID,

            REGEXP_REPLACE(
                person,
                r',\\d+$',
                ''
            ) AS Person

        FROM
            `gdelt-bq.gdeltv2.gkg_partitioned`,
            UNNEST(SPLIT(V2Persons, ';')) AS person

        WHERE
            _PARTITIONDATE BETWEEN DATE('{start_date}')
                                AND DATE('{end_date}')
            AND LOWER(V2Locations) LIKE '%{location.lower()}%'
            AND V2Persons IS NOT NULL
            AND V2Persons != ''

        ORDER BY
            Record_ID
    """

    return query_to_df(query)

In [59]:
generate_people_df()

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,Record_ID,Person
0,20260819131500-243,Ryan Mackenzie
1,20260819131500-243,Steve Ulrich
2,20260819131500-243,Scott Presler
3,20260819131500-243,Scott Presler
4,20260819131500-243,Nathan Benefield
...,...,...
1021,20260918050000-655,Michele Holly
1022,20260918050000-655,Payton Wilson
1023,20260918050000-655,Payton Wilson
1024,20260918050000-655,Curt Roberts


In [ ]:
# Function for generating the organizations dataframe
def generate_organizations_df(
        start_date='2026-08-19',
        end_date='2026-09-19',
        location='pittsburgh'
):

    query = f"""
        SELECT
            GKGRECORDID AS Record_ID,

            REGEXP_REPLACE(
                organization,
                r',\\d+$',
                ''
            ) AS Organization

        FROM
            `gdelt-bq.gdeltv2.gkg_partitioned`,
            UNNEST(SPLIT(V2Organizations, ';')) AS organization

        WHERE
            _PARTITIONDATE BETWEEN DATE('{start_date}')
                                AND DATE('{end_date}')
            AND LOWER(V2Locations) LIKE '%{location.lower()}%'
            AND V2Organizations IS NOT NULL
            AND V2Organizations != ''

        ORDER BY
            Record_ID
    """

    return query_to_df(query)

In [64]:
generate_organizations_df()

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,Record_ID,Organization
0,20260819131500-243,Little League Baseball World Series
1,20260819131500-243,Global Strategy Group
2,20260819131500-243,Pennsylvania Majority Fund Reps
3,20260819131500-243,Penn State University
4,20260819131500-243,Data Centers
...,...,...
1466,20260918050000-655,Major League Baseball
1467,20260918134500-675,Startup Program
1468,20260918134500-675,Nvidia
1469,20260918134500-675,Asics


In [65]:
# Function for generating the locations dataframe
def generate_locations_df(
        start_date='2026-08-19',
        end_date='2026-09-19',
        location='pittsburgh'
):

    query = f"""
        WITH location_data AS (

            SELECT
                GKGRECORDID AS Record_ID,

                location,

                SPLIT(location, '#') AS parts

            FROM
                `gdelt-bq.gdeltv2.gkg_partitioned`,
                UNNEST(SPLIT(V2Locations, ';')) AS location

            WHERE
                _PARTITIONDATE BETWEEN DATE('{start_date}')
                                    AND DATE('{end_date}')
                AND LOWER(V2Locations) LIKE '%{location.lower()}%'
                AND V2Locations IS NOT NULL
                AND V2Locations != ''
        )

        SELECT
            Record_ID,

            -- Creating GIS variable
            CONCAT(
                CAST(SAFE_CAST(parts[5] AS FLOAT64) AS STRING),
                ', ',
                CAST(SAFE_CAST(parts[6] AS FLOAT64) AS STRING)
            ) AS GIS,

            CASE
                WHEN SAFE_CAST(parts[0] AS INT64) IN (3, 4)
                THEN ARRAY_REVERSE(
                    SPLIT(parts[1], ', ')
                )[SAFE_OFFSET(2)]
                ELSE NULL
            END AS City,

            CASE
                WHEN SAFE_CAST(parts[0] AS INT64) IN (3, 4)
                THEN ARRAY_REVERSE(
                    SPLIT(parts[1], ', ')
                )[SAFE_OFFSET(1)]

                WHEN SAFE_CAST(parts[0] AS INT64) IN (2, 5)
                THEN ARRAY_REVERSE(
                    SPLIT(parts[1], ', ')
                )[SAFE_OFFSET(1)]

                ELSE NULL
            END AS State,

            ARRAY_REVERSE(
                SPLIT(parts[1], ', ')
            )[SAFE_OFFSET(0)] AS Country

        FROM
            location_data

        ORDER BY
            Record_ID
    """

    return query_to_df(query)


In [66]:
generate_locations_df()

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,Record_ID,GIS,City,State,Country
0,20260819131500-243,"41.1253, -80.0001",Butler County,Pennsylvania,United States
1,20260819131500-243,"41.1253, -80.0001",Butler County,Pennsylvania,United States
2,20260819131500-243,"39.828175, -98.5795",None,None,American
3,20260819131500-243,"32.5333, -117.017",Tijuana,Baja California,Mexico
4,20260819131500-243,"39.9523, -75.1638",Philadelphia,Pennsylvania,United States
...,...,...,...,...,...
2823,20260918134500-675,"40.5773, -77.264",None,Pennsylvania,United States
2824,20260918134500-675,"40.4406, -79.9959",Pittsburgh,Pennsylvania,United States
2825,20260918134500-675,"40.4406, -79.9959",Pittsburgh,Pennsylvania,United States
2826,20260918134500-675,"40.4406, -79.9959",Pittsburgh,Pennsylvania,United States


In [67]:
def generate_all_df(
        start_date='2026-08-19',
        end_date='2026-09-19',
        location='pittsburgh'
):

    df_main = generate_main_df(
        start_date,
        end_date,
        location
    )

    df_themes = generate_themes_df(
        start_date,
        end_date,
        location
    )

    df_people = generate_people_df(
        start_date,
        end_date,
        location
    )

    df_organizations = generate_organizations_df(
        start_date,
        end_date,
        location
    )

    df_locations = generate_locations_df(
        start_date,
        end_date,
        location
    )

    return (
        df_main,
        df_themes,
        df_people,
        df_organizations,
        df_locations
    )

In [121]:
df_main, df_themes, df_people, df_organizations, df_locations = generate_all_df(
    start_date='2026-09-01',
    end_date='2026-09-19',
    location='pittsburgh'
)

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [122]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   Record_ID       46 non-null     object             
 1   Date_Published  46 non-null     datetime64[us, UTC]
 2   Source_Name     46 non-null     object             
 3   Article_Link    46 non-null     object             
 4   Article_Title   46 non-null     object             
 5   Tone            46 non-null     object             
dtypes: datetime64[us, UTC](1), object(5)
memory usage: 2.3+ KB


In [123]:
df_main_cleaned = filter_working_articles(
    df_main,
    min_words= 300
)

In [124]:
df_main_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


In [120]:
df_main_cleaned.head()

""


Currently debugging filter because dataframes are coming up with no filters. For now, providing a small dataset for LLM person to work on detecting themes. Long file formatting is apparently not preferred, but maybe my previous functions are good for later? 😭 Or at least maybe the Location Table generated could be helpful. 

LLM person also requested JSON file instead of csv

5:01 pm: Debugged because of wrong variable name lol 

In [ ]:
query_llm = """
    SELECT
    GKGRECORDID AS Record_ID,

    PARSE_TIMESTAMP(
        '%Y%m%d%H%M%S',
        CAST(DATE AS STRING)
    ) AS Publication_Date,

    SourceCommonName AS Source_Name,

    V2Tone AS Tone,

    REGEXP_EXTRACT(
        Extras,
        r'<PAGE_TITLE>(.*?)</PAGE_TITLE>'
    ) AS Title,

    DocumentIdentifier AS Article_Link,

    V2Persons AS People,

    V2Organizations AS Organizations,

    V2Themes AS Themes

FROM
    `gdelt-bq.gdeltv2.gkg_partitioned`

WHERE
    _PARTITIONDATE BETWEEN DATE('2026-08-18')
                        AND DATE('2026-09-19')

    AND LOWER(V2Locations) LIKE '%pittsburgh%'

ORDER BY
    DATE DESC
            """

In [139]:
df_llm = query_to_df(query_llm)

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [146]:
df_llm.head()

,Record_ID,Publication_Date,Source_Name,Tone,Title,Article_Link,People,Organizations,Themes
0,20260918134500-675,2026-09-18 13:45:00+00:00,engineering.com,"4.4973544973545,5.29100529100529,0.79365079365...",NovoLINC launches MaxLINC thermal interface ma...,https://www.engineering.com/novolinc-launches-...,None,"Startup Program,2497;Nvidia,2516;Asics,1703;In...","EPU_ECONOMY_HISTORIC,456;EPU_ECONOMY_HISTORIC,..."
1,20260918050000-655,2026-09-18 05:00:00+00:00,birminghamstar.com,"3.99208775400108,5.25984535155548,1.2677575975...",Community Corner: Something special,http://www.birminghamstar.com/news/279314326/c...,"Steelers Hall,31537;Steelers Hall,61664;Alyssa...","University Of Alabama,60709;History Month Livi...",None
2,20260918000000-629,2026-09-18 00:00:00+00:00,northcentralpa.com,"1.05540897097625,3.69393139841689,2.6385224274...",Pa. election 2026: Your complete guide to the ...,https://www.northcentralpa.com/features/pa-ele...,"Josh Shapiro,361;Josh Shapiro,2181;Josh Shapir...","Siena University,4617;Pa Future Fund,12191;Ses...","EDUCATION,2497;EDUCATION,6939;EDUCATION,18269;..."
3,20260917211500-33,2026-09-17 21:15:00+00:00,prnewswire.com,"3.29861111111111,4.34027777777778,1.0416666666...","NovoLINC Launches MaxLINC, Achieving Industry-...",http://www.prnewswire.com/news-releases/novoli...,"Darren Goetz,2695","Inception Program,3244;Startup Program,3213;As...","WB_507_ENERGY_AND_EXTRACTIVES,4129;WB_533_ENER..."
4,20260917120000-543,2026-09-17 12:00:00+00:00,butlereagle.com,"-1.31061598951507,2.22804718217562,3.538663171...",Nonprofit claims ICE agent attempted to enter ...,https://www.butlereagle.com/20260916/nonprofit...,"Jaime Martinez,933;Mike Slupe,1481;Dan Santoro...","Instagram,784","PERSECUTION,4395;DISCRIMINATION,4395;TAX_MILIT..."


In [145]:
df_llm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   Record_ID         66 non-null     object             
 1   Publication_Date  66 non-null     datetime64[us, UTC]
 2   Source_Name       66 non-null     object             
 3   Tone              66 non-null     object             
 4   Title             66 non-null     object             
 5   Article_Link      66 non-null     object             
 6   People            64 non-null     object             
 7   Organizations     58 non-null     object             
 8   Themes            55 non-null     object             
dtypes: datetime64[us, UTC](1), object(8)
memory usage: 4.8+ KB


In [143]:
df_llm.to_json(
    "pittsburgh_json.json",
    orient="records",
    indent=2
)

In [153]:
df_llm_cleaned= filter_working_articles(df_llm)

In [154]:
df_llm_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   Record_ID         48 non-null     object             
 1   Publication_Date  48 non-null     datetime64[ns, UTC]
 2   Source_Name       48 non-null     object             
 3   Tone              48 non-null     object             
 4   Title             48 non-null     object             
 5   Article_Link      48 non-null     object             
 6   People            46 non-null     object             
 7   Organizations     46 non-null     object             
 8   Themes            44 non-null     object             
 9   Article_Text      48 non-null     object             
dtypes: datetime64[ns, UTC](1), object(9)
memory usage: 3.9+ KB


In [155]:
df_llm_cleaned.head()

,Record_ID,Publication_Date,Source_Name,Tone,Title,Article_Link,People,Organizations,Themes,Article_Text
0,20260918134500-675,2026-09-18 13:45:00+00:00,engineering.com,"4.4973544973545,5.29100529100529,0.79365079365...",NovoLINC launches MaxLINC thermal interface ma...,https://www.engineering.com/novolinc-launches-...,None,"Startup Program,2497;Nvidia,2516;Asics,1703;In...","EPU_ECONOMY_HISTORIC,456;EPU_ECONOMY_HISTORIC,...",MaxLINC targets heat flux above 100 W/cm² and ...
1,20260918050000-655,2026-09-18 05:00:00+00:00,birminghamstar.com,"3.99208775400108,5.25984535155548,1.2677575975...",Community Corner: Something special,http://www.birminghamstar.com/news/279314326/c...,"Steelers Hall,31537;Steelers Hall,61664;Alyssa...","University Of Alabama,60709;History Month Livi...",None,"The Steelers\n \n17 Sep 2026, 2..."
2,20260918000000-629,2026-09-18 00:00:00+00:00,northcentralpa.com,"1.05540897097625,3.69393139841689,2.6385224274...",Pa. election 2026: Your complete guide to the ...,https://www.northcentralpa.com/features/pa-ele...,"Josh Shapiro,361;Josh Shapiro,2181;Josh Shapir...","Siena University,4617;Pa Future Fund,12191;Ses...","EDUCATION,2497;EDUCATION,6939;EDUCATION,18269;...","Spotlight PA is an independent, nonpartisan, a..."
3,20260917211500-33,2026-09-17 21:15:00+00:00,prnewswire.com,"3.29861111111111,4.34027777777778,1.0416666666...","NovoLINC Launches MaxLINC, Achieving Industry-...",http://www.prnewswire.com/news-releases/novoli...,"Darren Goetz,2695","Inception Program,3244;Startup Program,3213;As...","WB_507_ENERGY_AND_EXTRACTIVES,4129;WB_533_ENER...","PITTSBURGH, Sept. 17, 2026 /PRNewswire/ -- Nov..."
4,20260917120000-543,2026-09-17 12:00:00+00:00,butlereagle.com,"-1.31061598951507,2.22804718217562,3.538663171...",Nonprofit claims ICE agent attempted to enter ...,https://www.butlereagle.com/20260916/nonprofit...,"Jaime Martinez,933;Mike Slupe,1481;Dan Santoro...","Instagram,784","PERSECUTION,4395;DISCRIMINATION,4395;TAX_MILIT...",Nonprofit claims ICE agent attempted to enter ...


In [156]:
df_llm_cleaned.to_json(
    "pittsburgh_json_cleaned.json",
    orient="records",
    indent=2
)

Since the dataset is really small for Pittsburgh, maybe we can scale and look at PA. Maybe considering to just look at one week instead of a month now

In [ ]:
query_pa_one_week = """
    SELECT
    GKGRECORDID AS Record_ID,

    PARSE_TIMESTAMP(
        '%Y%m%d%H%M%S',
        CAST(DATE AS STRING)
    ) AS Publication_Date,

    SourceCommonName AS Source_Name,

    V2Tone AS Tone,

    REGEXP_EXTRACT(
        Extras,
        r'<PAGE_TITLE>(.*?)</PAGE_TITLE>'
    ) AS Title,

    DocumentIdentifier AS Article_Link,

    V2Persons AS People,

    V2Organizations AS Organizations,

    V2Themes AS Themes

    FROM
        `gdelt-bq.gdeltv2.gkg_partitioned`

    WHERE
        _PARTITIONDATE BETWEEN DATE('2026-09-12')
                            AND DATE('2026-09-19')

        AND LOWER(V2Locations) LIKE '%pennsylvania%'

    ORDER BY
        DATE DESC
            """

In [171]:
df_pa_one_week = query_to_df(query_pa_one_week)

c:\Users\kellf\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [173]:
df_pa_one_week.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18984 entries, 0 to 18983
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   Record_ID         18984 non-null  object             
 1   Publication_Date  18984 non-null  datetime64[us, UTC]
 2   Source_Name       18984 non-null  object             
 3   Tone              18984 non-null  object             
 4   Title             18984 non-null  object             
 5   Article_Link      18984 non-null  object             
 6   People            16336 non-null  object             
 7   Organizations     16316 non-null  object             
 8   Themes            14569 non-null  object             
dtypes: datetime64[us, UTC](1), object(8)
memory usage: 1.3+ MB


In [ ]:
#making df into long format so can count themes
themes = (
    df_pa_one_week['Themes']
    .dropna()
    .str.split(';')
    .explode() #Transform each element of a list-like to a row, replicating index values; this looks cool
)

themes = (
    themes
    .str.replace(r',\d+$', '', regex=True)
    .str.strip()
)

theme_counts = themes.value_counts().reset_index()
theme_counts.columns = ['Theme', 'Count']

In [178]:
type(theme_counts)

pandas.core.frame.DataFrame

In [188]:
top_100_themes_pa_one_week = theme_counts.head(100)

In [ ]:
#Extracting top 100 articles for front end to understand the top themes in Pennsylvania
top_100_themes_pa_one_week.to_csv('top_100_themes_pa_one_week.csv', index=False)

In [ ]:
#took 195 m and 17s to process 18,000+ rows
df_pa_one_week_cleaned = filter_working_articles(df_pa_one_week)

In [192]:
df_pa_one_week_cleaned.to_json(
    "pa_one_week_cleaned_json.json",
    orient="records",
    indent=2
)

In [193]:
df_pa_one_week_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12650 entries, 0 to 12649
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   Record_ID         12650 non-null  object             
 1   Publication_Date  12650 non-null  datetime64[ns, UTC]
 2   Source_Name       12650 non-null  object             
 3   Tone              12650 non-null  object             
 4   Title             12650 non-null  object             
 5   Article_Link      12650 non-null  object             
 6   People            11452 non-null  object             
 7   Organizations     10876 non-null  object             
 8   Themes            9972 non-null   object             
 9   Article_Text      12650 non-null  object             
dtypes: datetime64[ns, UTC](1), object(9)
memory usage: 988.4+ KB


Since 12,650 rows is hard to process, decided to split data into daily for teammate to process in chunks

In [209]:
#Converting to datetime
df_pa_one_week_cleaned['Publication_Date'] = pd.to_datetime(
    df_pa_one_week_cleaned['Publication_Date']
)

#Grouping by calendar day
daily_dfs = {
    date: group.copy()
    for date, group in df_pa_one_week_cleaned.groupby(
        df_pa_one_week_cleaned['Publication_Date'].dt.date
    )
}

In [212]:
daily_dfs = {}

#Creating dataframes from September 12-19
for day in range(12, 20):
    date = pd.to_datetime(f'2026-09-{day:02d}').date()

    daily_dfs[date] = (
        df_pa_one_week_cleaned[
            df_pa_one_week_cleaned['Publication_Date'].dt.date == date
        ].copy()
    )

In [ ]:
#converting dataframes in Json files
for date, df in daily_dfs.items():
    df.to_json(
        f"df_{date}.json",
        orient="records",
        indent=2
    )